In [ ]:
import pandas as pd, numpy as np
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
from lsff_utils import paths, gbd_data
import pathlib

In [ ]:
location = "India"
vehicle = "rice"

In [ ]:
location = location.title()

In [ ]:
# Simulation output lives in per-run, timestamp-named directories, so the run
# is resolved rather than hardcoded -- this is the same run the Snakefile's
# marker names, so the notebook and the workflow cannot disagree.
maternal_results = paths.latest_results(paths.MATERNAL_RESULTS_ROOT, location, vehicle)
maternal_results

In [ ]:
with gbd_data.quiet_gbd_logs():
    pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

In [ ]:
with gbd_data.quiet_gbd_logs():
    asfr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location
    ).value
asfr[asfr > 0]

In [ ]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr

In [ ]:
with gbd_data.quiet_gbd_logs():
    sbr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.stillbirth_28_weeks_to_live_birth_ratio, "estimate", location
    ).value
sbr[sbr > 0]

In [ ]:
sbr = sbr.iloc[0]
sbr

In [ ]:
with gbd_data.quiet_gbd_logs():
    maternal_abortion_miscarriage_incidence = vivarium_inputs.get_measure(
        gbd_mapping.causes.maternal_abortion_and_miscarriage,
        "incidence_rate",
        location,
    ).mean(axis=1)
maternal_abortion_miscarriage_incidence[maternal_abortion_miscarriage_incidence > 0]

In [ ]:
with gbd_data.quiet_gbd_logs():
    ectopic_pregnancy_incidence = vivarium_inputs.get_measure(
        gbd_mapping.causes.ectopic_pregnancy,
        "incidence_rate",
        location,
    ).mean(axis=1)
ectopic_pregnancy_incidence[ectopic_pregnancy_incidence > 0]

In [ ]:
pregnancy_incidence = (
    asfr
    + (asfr * sbr)
    + maternal_abortion_miscarriage_incidence
    + ectopic_pregnancy_incidence
)
pregnancy_incidence[pregnancy_incidence > 0]

In [ ]:
pregnancies = pop * pregnancy_incidence
pregnancies[pregnancies > 0]

In [ ]:
total_pregnancies = pregnancies.sum()
total_pregnancies

In [ ]:
sim_population = (
    pd.read_parquet(
        paths.measure_path(maternal_results, "pregnancy_outcome_count")
    )
    .groupby(["input_draw", "scenario"])
    .value.sum()
    .mean()
)
sim_population

In [ ]:
scalar = total_pregnancies / sim_population
scalar

In [ ]:
for result in [
    "ylds",
    "ylls",
    "pregnancy_outcome_count",
    "person_time_anemia",
    "transition_count_maternal_disorders",
]:
    df = pd.read_parquet(
        paths.measure_path(maternal_results, result)
    )
    df.value *= scalar
    path = pathlib.Path(
        f"../results/rescaled_pregnancy_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)